# Pilot · AR verifiers on the pilot bank (T1, T2, T3)

Runtime: **A100**. One fresh runtime. Colab secret `HF_TOKEN` with access to Llama, Gemma and Mistral.

Scores the 500 pilot answers with Qwen2.5-7B, Mistral-7B, Llama-3.1-8B, Gemma-2-9B, Llama-3.2-3B and Qwen2.5-1.5B under all three templates (gate B needs T1–T3). Labels are not read here. About 30–40 minutes, mostly model loading.

In [ ]:
BRANCH = "feature/pilot-bank"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r requirements.txt -r requirements-analysis.txt
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
OUT = "results/pilot/ar"
!mkdir -p {OUT}
import hashlib
PILOT_HASH = hashlib.sha256(open("bank/pilot/pilot_bank.jsonl", "rb").read()).hexdigest()
print("pilot bank", PILOT_HASH)

In [ ]:
for cfg in ("pilot_qwen_mistral", "pilot_llama_gemma", "pilot_small"):
    for t in ("T1", "T2", "T3"):
        !python -m harness.run --config configs/{cfg}.json --input bank/pilot/pilot_bank.jsonl --template {t} --output-dir {OUT}/{cfg}_{t} --device cuda:0 --dtype bf16 --no-load-in-4bit

## Download the results
Zips `results/pilot/ar/`. Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/pilot_ar.zip results/pilot/ar
from google.colab import files
files.download("/content/pilot_ar.zip")